# Verification: the `rest3.3-3.5.pdf` update (§3.3 arc length, §3.5 volume)

This notebook supports the 2026-09-25 update of `study-guides/calc2_study_guide_1.1-3.3.html`. The filename is historical; the guide now runs through §3.5. The update added:
- the lecture notes `materials/notes/rest3.3-3.5.pdf` (7 pages);
- a new §3.3 motion practice problem (notes p. 1);
- a new **§3.3 Arc length** section (notes pp. 2–4);
- a new **§3.5 Volume: disks and washers** section (notes pp. 5–7);
- new diagnostic, mixed-practice and final-check items.

Method, as in the earlier notebook (`verify_ftc_improper_numerical_displacement.ipynb`):
1. Every answer was derived **by hand** first (the `claim` column).
2. **SymPy** recomputes each one **from the problem statement**, never from the answer, and **mpmath** checks it numerically, independently.
3. Every targeted wrong-answer value is recomputed from the mistake it describes. Numbers quoted in explanations, tryouts, spot-the-error items, figures and explorers are checked too.
4. **Wolfram Language** results, computed from the problem statements, are pasted verbatim and compared.
5. The **built HTML** is audited. Every new answer must be verified. Every answer that existed before the update must be unchanged, checked against `guide_state_before_2026-09-25.json`. Nothing may be lost, and the scope is scanned.

"3.3–3.5 notes p. N" means `rest3.3-3.5.pdf`, page N.

In [1]:
import sympy as sp, mpmath as mp, json, re, subprocess, pathlib, html as _html
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
mp.mp.dps = 30
X, T, Y, W, B = sp.symbols('x t y w b', positive=True)
ROOT = pathlib.Path(r"C:\Users\chris\MathStudy")
BUILD = ROOT / "study-guides" / "_build" / "calc2_ftc_improper_numerical_displacement"
GUIDE = ROOT / "study-guides" / "calc2_study_guide_1.1-3.3.html"
VER = ROOT / "study-guides" / "verification"
ROWS, VERIFIED = [], {}

def fnum(e):
    return 'diverges' if isinstance(e, str) else float(sp.N(e, 25))

def check(vid, claim, sym_val, numeric_val=None, rel=1e-10, note=''):
    """claim = hand-derived answer; sym_val = SymPy from the problem statement; numeric_val = independent mpmath value."""
    c, s_ = fnum(claim), fnum(sym_val)
    n_ = None if numeric_val is None else float(numeric_val)
    same = lambda a_, b_: abs(a_ - b_) <= rel * max(1.0, abs(a_), abs(b_))
    ok = same(c, s_) and (n_ is None or same(c, n_))
    ROWS.append(dict(id=vid, claim=str(claim), sympy=str(sym_val), numeric=n_, ok=ok, note=note))
    VERIFIED[vid] = c
    print(f"{'OK      ' if ok else 'MISMATCH'} {vid:<18} claim={str(claim):<30} sympy={str(sp.nsimplify(sym_val)):<32} mpmath={'' if n_ is None else f'{n_:.15g}'}  {note}")
    return ok

def arc_graph(f, a, b, v=X):
    """Arc length of y = f(v), a <= v <= b (3.3-3.5 notes p. 2): the integral of sqrt(1 + f'(v)^2)."""
    return sp.integrate(sp.sqrt(sp.factor(sp.simplify(1 + sp.diff(f, v)**2))), (v, a, b))

def arc_param(P, a, b, v=T):
    """Distance traveled along (x(t), y(t)): the integral of the speed (3.3-3.5 notes p. 2)."""
    return sp.integrate(sp.simplify(sp.sqrt(sp.simplify(sum(sp.diff(c_, v)**2 for c_ in P)))), (v, a, b))

def washer(Ro, ri, v, a, b):
    """Washer method (3.3-3.5 notes p. 7): pi times the integral of R^2 - r^2 (r = 0 gives disks)."""
    return sp.pi * sp.integrate(sp.expand(Ro**2 - ri**2), (v, a, b))

def trap_rule(f, a, b, n):
    h = (b - a) / n
    return h / 2 * (f(a) + f(b) + 2 * sum(f(a + k * h) for k in range(1, n)))

def simp_rule(f, a, b, n):
    h = (b - a) / n
    return h / 3 * (f(a) + f(b) + sum((4 if k % 2 else 2) * f(a + k * h) for k in range(1, n)))

def improper(expr, v, a):
    """Type I improper integral by the notes' definition (notes p. 27): integrate to b, then let b -> oo."""
    return sp.limit(sp.integrate(expr, (v, a, B)), B, sp.oo)

mq = lambda f, pts: mp.quad(f, pts)
print("setup ok")

setup ok


## 1. The lecture's own worked answers, and the examples it leaves blank

In [2]:
print("== rest3.3-3.5.pdf: the professor's worked answers ==")
sp2 = sp.expand((T + 3)**2 + (sp.sqrt(2*T))**2 + 7)
print("   p. 1: |v|^2 =", sp2, "=", sp.factor(sp2))
check('n3-p1-dist', 28, sp.integrate(sp.sqrt(sp.factor(sp2)), (T, 1, 5)), mq(lambda s: mp.sqrt((s + 3)**2 + 2*s + 7), [1, 5]), note='notes p. 1 writes 28 (m)')
g = sp.Rational(2, 3) * (X + 4)**sp.Rational(3, 2)
print("   p. 3: g'(x) =", sp.simplify(sp.diff(g, X)), "   1 + g'(x)^2 =", sp.simplify(1 + sp.diff(g, X)**2))
check('n3-p3-g', sp.Rational(74, 3), arc_graph(g, 4, 11), mq(lambda s: mp.sqrt(1 + mp.diff(lambda z: mp.mpf(2)/3*(z + 4)**1.5, s)**2), [4, 11]), note='notes p. 3 writes 74/3')
check('n3-p3-semi', sp.pi, arc_param([sp.cos(T), sp.sin(T)], 0, sp.pi), mq(lambda s: mp.sqrt(mp.diff(mp.cos, s)**2 + mp.diff(mp.sin, s)**2), [0, mp.pi]), note='notes p. 3 writes pi')
T4 = trap_rule(lambda s: sp.sqrt(1 + sp.cos(s)**2), 0, 2*sp.pi, 4)
check('n3-p4-T4', sp.pi*(sp.sqrt(2) + 1), sp.simplify(T4), trap_rule(lambda s: mp.sqrt(1 + mp.cos(s)**2), 0, 2*mp.pi, 4), note='notes p. 4 writes pi(sqrt 2 + 1)')
true_sin = mq(lambda s: mp.sqrt(1 + mp.cos(s)**2), [0, mp.pi/2, mp.pi, 3*mp.pi/2, 2*mp.pi])
print("   p. 4: actual length =", mp.nstr(true_sin, 16), "| elliptic closed form 4*sqrt(2)*E(1/2) =", mp.nstr(4*mp.sqrt(2)*mp.ellipe(mp.mpf(1)/2), 16), "(no elementary antiderivative)")
print("== rest3.3-3.5.pdf: the two volume examples left blank (pp. 6-7), solved in the guide ==")
check('n3-p6-cone', 32*sp.pi/3, washer(2*X, 0, X, 0, 2), mq(lambda s: mp.pi*(2*s)**2, [0, 2]), note='disks, R = 2x')
check('n3-p6-coneformula', 32*sp.pi/3, sp.pi*4**2*2/sp.Integer(3), note='cone formula (1/3) pi r^2 h with r = 4, h = 2')
check('n3-p7-washer', 8*sp.pi, washer(2*X, X, X, 0, 2), mq(lambda s: mp.pi*((2*s)**2 - s**2), [0, 2]), note='washers, R = 2x, r = x')
check('n3-p7-cones', 8*sp.pi, sp.pi*4**2*2/sp.Integer(3) - sp.pi*2**2*2/sp.Integer(3), note='outer cone minus inner cone')

== rest3.3-3.5.pdf: the professor's worked answers ==
   p. 1: |v|^2 = t**2 + 8*t + 16 = (t + 4)**2
OK       n3-p1-dist         claim=28                             sympy=28                               mpmath=28  notes p. 1 writes 28 (m)


   p. 3: g'(x) = sqrt(x + 4)    1 + g'(x)^2 = x + 5
OK       n3-p3-g            claim=74/3                           sympy=74/3                             mpmath=24.6666666666667  notes p. 3 writes 74/3
OK       n3-p3-semi         claim=pi                             sympy=pi                               mpmath=3.14159265358979  notes p. 3 writes pi
OK       n3-p4-T4           claim=pi*(1 + sqrt(2))               sympy=pi*(1 + sqrt(2))                 mpmath=7.58447559174816  notes p. 4 writes pi(sqrt 2 + 1)
   p. 4: actual length = 7.640395578055424 | elliptic closed form 4*sqrt(2)*E(1/2) = 7.640395578055424 (no elementary antiderivative)
== rest3.3-3.5.pdf: the two volume examples left blank (pp. 6-7), solved in the guide ==
OK       n3-p6-cone         claim=32*pi/3                        sympy=32*pi/3                          mpmath=33.5103216382911  disks, R = 2x
OK       n3-p6-coneformula  claim=32*pi/3                        sympy=32*pi/3                          mpmath=  cone 

True

## 2. Every new numeric answer in the guide

These cover §3.3 arc length (the L section, mixed practice and the final check), §3.5 volume (the V section, mixed practice and the final check), and the new §3.3 motion problem.

In [3]:
print("== §3.3 arc length ==")
check('L-guided-val', sp.Rational(74, 3), arc_graph(g, 4, 11), mq(lambda s: mp.sqrt(s + 5), [4, 11]))
check('L-indep', sp.Rational(49, 3), arc_graph(sp.Rational(4, 3)*(X + 2)**sp.Rational(3, 2), 0, 4), mq(lambda s: mp.sqrt(1 + (2*mp.sqrt(s + 2))**2), [0, 4]), note='textbook 3.3 Suggested Ex. 6')
check('L-p1', sp.pi, arc_param([sp.cos(T), sp.sin(T)], 0, sp.pi), mq(lambda s: 1, [0, mp.pi]), note='notes p. 3')
check('L-p2', sp.pi*(sp.sqrt(2) + 1), sp.simplify(T4), trap_rule(lambda s: mp.sqrt(1 + mp.cos(s)**2), 0, 2*mp.pi, 4), note='notes p. 4, Trapezoid n = 4')
check('L-p3', sp.Rational(52, 3), arc_graph(sp.Rational(2, 3)*X**sp.Rational(3, 2), 0, 8), mq(lambda s: mp.sqrt(1 + s), [0, 8]), note='textbook Ex. 3.3.15')
f7 = X**3/12 + 1/X
print("   Suggested Ex. 7: 1 + f'^2 =", sp.factor(sp.simplify(1 + sp.diff(f7, X)**2)), " (a perfect square)")
check('L-p4', sp.Rational(13, 12), arc_graph(f7, 1, 2), mq(lambda s: mp.sqrt(1 + (s**2/4 - 1/s**2)**2), [1, 2]), note='textbook Suggested Ex. 7')
check('L-p5-a', 6*sp.pi, arc_param([3*sp.cos(T), 3*sp.sin(T)], 0, 2*sp.pi), mq(lambda s: mp.sqrt((3*mp.sin(s))**2 + (3*mp.cos(s))**2), [0, 2*mp.pi]), note='textbook Ex. 3.3.17, r = 3')
check('L-p5-b', 12*sp.pi, arc_param([3*sp.cos(2*T), 3*sp.sin(2*T)], 0, 2*sp.pi), mq(lambda s: mp.sqrt((6*mp.sin(2*s))**2 + (6*mp.cos(2*s))**2), [0, 2*mp.pi]), note='goes around twice')
check('L-p6', 4*sp.sqrt(10), arc_graph(3*X + 1, 1, 5), mp.sqrt(4**2 + 12**2), note='= the chord from (1, 4) to (5, 16)')
hand_S4 = sp.Rational(1, 12)*(1 + 4*sp.sqrt(265)/16 + 2*sp.Rational(5, 4) + 4*sp.sqrt(985)/16 + sp.sqrt(10))   # heights worked by hand
check('L-p7', hand_S4, simp_rule(lambda s: sp.sqrt(1 + 9*s**4), 0, sp.Integer(1), 4), simp_rule(lambda s: mp.sqrt(1 + 9*s**4), 0, mp.mpf(1), 4), note='Simpson n = 4; true value ' + mp.nstr(mq(lambda s: mp.sqrt(1 + 9*s**4), [0, 1]), 10))
check('L-p8', (13*sp.sqrt(13) - 8)/27, arc_param([T**2, T**3], 0, 1), mq(lambda s: mp.sqrt(4*s**2 + 9*s**4), [0, 1]))
check('L-m3-a', 3*sp.pi, arc_param([sp.cos(3*T), sp.sin(3*T)], 0, sp.pi), mq(lambda s: 3, [0, mp.pi]), note='distance traveled')
check('L-m3-b', 2*sp.pi, arc_param([sp.cos(T), sp.sin(T)], 0, 2*sp.pi), mq(lambda s: 1, [0, 2*mp.pi]), note='the angle 3t covers [0, 3 pi]: the whole unit circle is traced')
check('L-m4', sp.E**2/2 - sp.Rational(1, 4), arc_graph(X**2/2 - sp.log(X)/4, 1, sp.E), mq(lambda s: mp.sqrt(1 + (s - 1/(4*s))**2), [1, mp.e]))
check('F-L2', sp.Rational(14, 3), arc_graph(sp.Rational(2, 3)*(X - 1)**sp.Rational(3, 2), 1, 4), mq(lambda s: mp.sqrt(1 + (s - 1)), [1, 4]))
ast_id = sp.simplify(sp.diff(sp.cos(T)**3, T)**2 + sp.diff(sp.sin(T)**3, T)**2 - (3*sp.sin(T)*sp.cos(T))**2)
print("   astroid: speed^2 - (3 sin t cos t)^2 =", ast_id, "(and 3 sin t cos t >= 0 on [0, pi/2])")
check('F-L3', sp.Rational(3, 2), sp.integrate(3*sp.sin(T)*sp.cos(T), (T, 0, sp.pi/2)), mq(lambda s: mp.sqrt((3*mp.cos(s)**2*mp.sin(s))**2 + (3*mp.sin(s)**2*mp.cos(s))**2), [0, mp.pi/2]))
fm = (X**2 + 2)**sp.Rational(3, 2)/3
print("   mixed L1: 1 + f'^2 =", sp.factor(sp.simplify(1 + sp.diff(fm, X)**2)))
check('M-L1', 12, arc_graph(fm, 0, 3), mq(lambda s: mp.sqrt(1 + s**2*(s**2 + 2)), [0, 3]))
T4c = trap_rule(lambda s: sp.sqrt(1 + sp.sin(s)**2), 0, sp.pi, 4)
check('M-L2', sp.pi/4*(1 + sp.sqrt(2) + sp.sqrt(6)), sp.simplify(T4c), trap_rule(lambda s: mp.sqrt(1 + mp.sin(s)**2), 0, mp.pi, 4), note='Trapezoid n = 4; true value ' + mp.nstr(mq(lambda s: mp.sqrt(1 + mp.sin(s)**2), [0, mp.pi]), 10))

== §3.3 arc length ==
OK       L-guided-val       claim=74/3                           sympy=74/3                             mpmath=24.6666666666667  
OK       L-indep            claim=49/3                           sympy=49/3                             mpmath=16.3333333333333  textbook 3.3 Suggested Ex. 6
OK       L-p1               claim=pi                             sympy=pi                               mpmath=3.14159265358979  notes p. 3
OK       L-p2               claim=pi*(1 + sqrt(2))               sympy=pi*(1 + sqrt(2))                 mpmath=7.58447559174816  notes p. 4, Trapezoid n = 4
OK       L-p3               claim=52/3                           sympy=52/3                             mpmath=17.3333333333333  textbook Ex. 3.3.15
   Suggested Ex. 7: 1 + f'^2 = (x**2 - 2*x + 2)**2*(x**2 + 2*x + 2)**2/(16*x**4)  (a perfect square)


OK       L-p4               claim=13/12                          sympy=13/12                            mpmath=1.08333333333333  textbook Suggested Ex. 7
OK       L-p5-a             claim=6*pi                           sympy=6*pi                             mpmath=18.8495559215388  textbook Ex. 3.3.17, r = 3
OK       L-p5-b             claim=12*pi                          sympy=12*pi                            mpmath=37.6991118430775  goes around twice
OK       L-p6               claim=4*sqrt(10)                     sympy=4*sqrt(10)                       mpmath=12.6491106406735  = the chord from (1, 4) to (5, 16)
OK       L-p7               claim=sqrt(10)/12 + 7/24 + sqrt(265)/48 + sqrt(985)/48 sympy=sqrt(10)/12 + 7/24 + sqrt(265)/48 + sqrt(985)/48 mpmath=1.54818001853591  Simpson n = 4; true value 1.547865655


OK       L-p8               claim=-8/27 + 13*sqrt(13)/27         sympy=-8/27 + 13*sqrt(13)/27           mpmath=1.43970987337155  
OK       L-m3-a             claim=3*pi                           sympy=3*pi                             mpmath=9.42477796076938  distance traveled
OK       L-m3-b             claim=2*pi                           sympy=2*pi                             mpmath=6.28318530717959  the angle 3t covers [0, 3 pi]: the whole unit circle is traced
OK       L-m4               claim=-1/4 + exp(2)/2                sympy=-1/4 + exp(2)/2                  mpmath=3.44452804946533  
OK       F-L2               claim=14/3                           sympy=14/3                             mpmath=4.66666666666667  
   astroid: speed^2 - (3 sin t cos t)^2 = 0 (and 3 sin t cos t >= 0 on [0, pi/2])
OK       F-L3               claim=3/2                            sympy=3/2                              mpmath=1.5  


   mixed L1: 1 + f'^2 = (x**2 + 1)**2
OK       M-L1               claim=12                             sympy=12                               mpmath=12  
OK       M-L2               claim=pi*(1 + sqrt(2) + sqrt(6))/4   sympy=pi*(1 + sqrt(2) + sqrt(6))/4     mpmath=3.81994364317984  Trapezoid n = 4; true value 3.820197789


True

In [4]:
print("== §3.5 volume ==")
check('V-guided-val', 8*sp.pi, washer(2*X, X, X, 0, 2), mq(lambda s: mp.pi*3*s**2, [0, 2]), note='notes p. 7')
check('V-indep', 3*sp.pi/10, washer(sp.sqrt(X), X**2, X, 0, 1), mq(lambda s: mp.pi*(s - s**4), [0, 1]))
check('V-p1', 32*sp.pi/3, washer(2*X, 0, X, 0, 2), mq(lambda s: mp.pi*4*s**2, [0, 2]), note='notes p. 6')
check('V-p2', 32*sp.pi/5, washer(X**2, 0, X, 0, 2), mq(lambda s: mp.pi*s**4, [0, 2]), note='textbook Ex. 3.5.7')
check('V-p3', 2*sp.pi/15, washer(X, X**2, X, 0, 1), mq(lambda s: mp.pi*(s**2 - s**4), [0, 1]), note='textbook Ex. 3.5.8')
check('V-p4', 8*sp.pi/15, washer(2 - X**2, 2 - X, X, 0, 1), mq(lambda s: mp.pi*((2 - s**2)**2 - (2 - s)**2), [0, 1]), note='textbook Ex. 3.5.9, axis y = 2')
check('V-p5', sp.pi/6, washer(sp.sqrt(Y), Y, Y, 0, 1), mq(lambda s: mp.pi*(s - s**2), [0, 1]), note='textbook Ex. 3.5.10, the y-axis')
check('V-p6', sp.Rational(1, 5), sp.integrate((X**2)**2, (X, 0, 1)), mq(lambda s: s**4, [0, 1]), note='square cross-sections, side x^2')
check('V-p7', sp.pi/8, sp.integrate(sp.pi*(sp.sqrt(X)/2)**2, (X, 0, 1)), mq(lambda s: mp.pi*s/4, [0, 1]), note='disk cross-sections with diameter sqrt x')
check('V-p8-a', sp.pi/3, washer(1, X**sp.Rational(1, 4), X, 0, 1), mq(lambda s: mp.pi*(1 - mp.sqrt(s)), [0, 1]))
check('V-p8-b', sp.pi/9, washer(Y**4, 0, Y, 0, 1), mq(lambda s: mp.pi*s**8, [0, 1]))
check('V-p9', 58, sp.integrate(3*X**2 + 1, (X, 2, 4)), mq(lambda s: 3*s**2 + 1, [2, 4]))
print("   V-p10: 4x - x^2 = 3 at x =", sp.solve(sp.Eq(4*X - X**2, 3), X))
check('V-p10', 104*sp.pi/15, washer(6 - 3, 6 - (4*X - X**2), X, 1, 3), mq(lambda s: mp.pi*(9 - (6 - 4*s + s**2)**2), [1, 3]), note='axis y = 6')
check('V-m3', sp.pi, sp.pi*improper(1/X**2, X, 1), mq(lambda s: mp.pi/s**2, [1, mp.inf]), note="Gabriel's horn, improper")
check('V-m4', 36*sp.pi, sp.pi*sp.integrate(9 - X**2, (X, -3, 3)), mq(lambda s: mp.pi*(9 - s**2), [-3, 3]), note='ball of radius 3 = (4/3) pi 3^3')
check('F-V2', 40*sp.pi/3, washer(2, 2 - sp.sqrt(X), X, 0, 4), mq(lambda s: mp.pi*(4 - (2 - mp.sqrt(s))**2), [0, 4]), note='axis y = 2, above the region')
check('F-V3', 2*sp.pi, sp.pi*improper(X**sp.Rational(-3, 2), X, 1), mq(lambda s: mp.pi*s**(-1.5), [1, mp.inf]), note='improper; the region itself has infinite area')
check('M-V1', 8*sp.pi, washer(sp.sqrt(X), 0, X, 0, 4), mq(lambda s: mp.pi*s, [0, 4]))
print("   M-V2: 3x - x^2 = x at x =", sp.solve(sp.Eq(3*X - X**2, X), X))
check('M-V2', 136*sp.pi/15, washer(3*X - X**2 + 2, X + 2, X, 0, 2), mq(lambda s: mp.pi*((3*s - s**2 + 2)**2 - (s + 2)**2), [0, 2]), note='axis y = -2')
check('M-V3', sp.pi/2, sp.pi*improper(sp.exp(-2*X), X, 0), mq(lambda s: mp.pi*mp.e**(-2*s), [0, mp.inf]), note='improper')
print("== §3.3 motion: the new practice problem (3.3-3.5 notes p. 1) ==")
check('E-p11', 28, sp.integrate(sp.sqrt(sp.factor(sp2)), (T, 1, 5)), mq(lambda s: mp.sqrt((s + 3)**2 + 2*s + 7), [1, 5]))

== §3.5 volume ==
OK       V-guided-val       claim=8*pi                           sympy=8*pi                             mpmath=25.1327412287183  notes p. 7
OK       V-indep            claim=3*pi/10                        sympy=3*pi/10                          mpmath=0.942477796076938  
OK       V-p1               claim=32*pi/3                        sympy=32*pi/3                          mpmath=33.5103216382911  notes p. 6
OK       V-p2               claim=32*pi/5                        sympy=32*pi/5                          mpmath=20.1061929829747  textbook Ex. 3.5.7
OK       V-p3               claim=2*pi/15                        sympy=2*pi/15                          mpmath=0.418879020478639  textbook Ex. 3.5.8
OK       V-p4               claim=8*pi/15                        sympy=8*pi/15                          mpmath=1.67551608191456  textbook Ex. 3.5.9, axis y = 2
OK       V-p5               claim=pi/6                           sympy=pi/6                             mpmath=0.5

OK       V-p7               claim=pi/8                           sympy=pi/8                             mpmath=0.392699081698724  disk cross-sections with diameter sqrt x
OK       V-p8-a             claim=pi/3                           sympy=pi/3                             mpmath=1.0471975511966  


OK       V-p8-b             claim=pi/9                           sympy=pi/9                             mpmath=0.349065850398866  
OK       V-p9               claim=58                             sympy=58                               mpmath=58  
   V-p10: 4x - x^2 = 3 at x = [1, 3]
OK       V-p10              claim=104*pi/15                      sympy=104*pi/15                        mpmath=21.7817090648892  axis y = 6
OK       V-m3               claim=pi                             sympy=pi                               mpmath=3.14159265358979  Gabriel's horn, improper
OK       V-m4               claim=36*pi                          sympy=36*pi                            mpmath=113.097335529233  ball of radius 3 = (4/3) pi 3^3
OK       F-V2               claim=40*pi/3                        sympy=40*pi/3                          mpmath=41.8879020478639  axis y = 2, above the region
OK       F-V3               claim=2*pi                           sympy=2*pi                            

OK       M-V3               claim=pi/2                           sympy=pi/2                             mpmath=1.5707963267949  improper
== §3.3 motion: the new practice problem (3.3-3.5 notes p. 1) ==
OK       E-p11              claim=28                             sympy=28                               mpmath=28  


True

## 3. Typed-expression answers (guided steps), checked with SymPy and with the guide's own grader

The intended expression is derived from the problem, not copied from the key. `js_eval.js` extracts the parser and grader from `app.js` and runs them on every expression key in the guide, old and new. It checks three things:
- the reference answer is accepted;
- a typed `+ C` is accepted where it applies;
- each listed wrong answer is rejected with its own targeted message.

In [5]:
TR = standard_transformations + (implicit_multiplication_application, convert_xor)
FUN = {'e': sp.E, 'pi': sp.pi, 'ln': sp.log, 'sqrt': sp.sqrt, 'abs': sp.Abs}
js2sp = lambda s_: parse_expr(s_, local_dict={**FUN, 'x': X}, transformations=TR)
KEYS = {k_['vid']: k_ for k_ in json.load(open(BUILD / 'answer_keys_full.json', encoding='utf-8'))}
EXPR_REF = {                                   # the intended expression, derived from each problem
    'L-guided-dg': sp.diff(g, X),              # g'(x) for g = (2/3)(x + 4)^(3/2)
    'L-guided-inside': sp.simplify(1 + sp.diff(g, X)**2),
    'V-guided-R': 2*X - 0,                     # far edge y = 2x, measured from the axis y = 0
    'V-guided-r': X - 0,                       # near edge y = x
    'V-guided-A': sp.pi*((2*X)**2 - X**2),     # pi (R^2 - r^2)
}
EXPR_OK = {}
for vid, ref in EXPR_REF.items():
    k_ = KEYS[vid]
    ok = sp.simplify(js2sp(k_['answer']) - ref) == 0
    wrong_differ = all(sp.simplify(js2sp(w_['f']) - ref) != 0 for w_ in k_['wrong'] if 'f' in w_)
    EXPR_OK[vid] = ok and wrong_differ
    print(f"{'OK      ' if EXPR_OK[vid] else 'MISMATCH'} {vid:<16} key = {k_['answer']:<12} intended = {str(ref):<12} every listed wrong expression differs: {wrong_differ}")
JSRUN = subprocess.run(['node', str(BUILD / 'js_eval.js')], capture_output=True, text=True)
print("\nthe guide's own parser and grader on ALL expression keys:", JSRUN.stdout.strip() or JSRUN.stderr)
JSV = {r_['vid']: r_ for r_ in json.load(open(BUILD / 'js_eval.json', encoding='utf-8'))}
for vid, ref in EXPR_REF.items():
    r_ = JSV[vid]
    agree = all(abs(float(ref.subs(X, p_)) - v_) < 1e-9 * max(1, abs(v_)) for p_, v_ in zip(r_['pts'], r_['answer']))
    targeted = all((not w_['ok']) and w_['targeted'] for w_ in r_['wrong'])
    EXPR_OK[vid] = EXPR_OK[vid] and agree and r_['self_ok'] and targeted
    print(f"   {vid:<16} JS values = SymPy values: {agree} | reference accepted: {r_['self_ok']} | each wrong answer rejected with its message: {targeted}")
bad_js = [r_['vid'] for r_ in JSV.values() if not r_['self_ok'] or not r_['withC_ok'] or any(w_['ok'] or not w_['targeted'] for w_ in r_['wrong'])]
print("expression keys (old or new) where the grader misbehaves:", bad_js or 'none')

OK       L-guided-dg      key = (x+4)^(1/2)  intended = sqrt(x + 4)  every listed wrong expression differs: True
OK       L-guided-inside  key = x+5          intended = x + 5        every listed wrong expression differs: True
OK       V-guided-R       key = 2*x          intended = 2*x          every listed wrong expression differs: True
OK       V-guided-r       key = x            intended = x            every listed wrong expression differs: True
OK       V-guided-A       key = 3*pi*x^2     intended = 3*pi*x**2    every listed wrong expression differs: True

the guide's own parser and grader on ALL expression keys: expression keys: 55 | reference accepted: 55 | with +C accepted: 55 | wrong answers rejected with their targeted message: 31/31
   L-guided-dg      JS values = SymPy values: True | reference accepted: True | each wrong answer rejected with its message: True
   L-guided-inside  JS values = SymPy values: True | reference accepted: True | each wrong answer rejected with its me

## 4. Targeted wrong-answer values: each one really is what the described mistake produces

In [6]:
MIST = {
 ('E-p11', 27.930697094774): (sp.sqrt(sum(sp.integrate(c_, (T, 1, 5))**2 for c_ in [T + 3, sp.sqrt(2*T), sp.sqrt(7)])), '|displacement|'),
 ('E-p11', 44.181121736571): (sp.integrate(T + 3 + sp.sqrt(2*T) + sp.sqrt(7), (T, 1, 5)), 'speed taken as the sum of the components'),
 ('L-guided-val', 18.98858179594): (sp.Rational(2, 3)*(11**sp.Rational(3, 2) - 4**sp.Rational(3, 2)), 'x-limits 4, 11 put into (2/3)u^(3/2)'),
 ('L-indep', 65.333333333333): (sp.Rational(2, 3)*(25**sp.Rational(3, 2) - 9**sp.Rational(3, 2)), 'forgot du = 4 dx'),
 ('L-indep', 11.897275564511): (sp.integrate(sp.sqrt(2*X + 5), (X, 0, 4)), "g'^2 taken as 2(x + 2)"),
 ('L-p1', 6.28318530718): (2*sp.pi, 'the whole circle'),
 ('L-p1', 2.0): (sp.sqrt((1 - (-1))**2 + 0), 'the chord from (1, 0) to (-1, 0)'),
 ('L-p2', 5.363034122669): (sp.pi/4*(sp.sqrt(2) + 2 + 0 + 2 + sp.sqrt(2)), 'f(pi) taken as 0'),
 ('L-p2', 15.168951183496): (sp.pi/2*(sp.sqrt(2) + 2 + 2*sp.sqrt(2) + 2 + sp.sqrt(2)), 'prefactor dx instead of dx/2'),
 ('L-p2', 3.14159265359): (sp.pi/4*(1 + 2*0 + 2*1 + 2*0 + 1), 'heights |cos x| (the 1 under the root missing)'),
 ('L-p5-b', 18.849555921539): (6*sp.pi, "the circle's length, not the distance"),
 ('L-p7', 1.577527367662): (trap_rule(lambda s: sp.sqrt(1 + 9*s**4), 0, sp.Integer(1), 4), 'Trapezoid instead of Simpson'),
 ('L-p8', 2.0): (sp.integrate(2*T + 3*T**2, (T, 0, 1)), 'root split: 2t + 3t^2'),
 ('L-m3-b', 9.424777960769): (3*sp.pi, 'the distance traveled'),
 ('L-m3-b', 3.14159265359): (sp.pi, 'only half the circle'),
 ('V-guided-val', 8.377580409573): (sp.pi*sp.integrate((2*X - X)**2, (X, 0, 2)), 'pi (R - r)^2'),
 ('V-guided-val', 33.510321638291): (sp.pi*sp.integrate((2*X)**2, (X, 0, 2)), 'no hole (the whole cone)'),
 ('V-guided-val', 8.0): (sp.integrate(3*X**2, (X, 0, 2)), 'pi forgotten'),
 ('V-indep', 0.403919055462): (sp.pi*sp.integrate((sp.sqrt(X) - X**2)**2, (X, 0, 1)), 'pi (R - r)^2'),
 ('V-indep', 1.570796326795): (sp.pi*sp.integrate(X, (X, 0, 1)), 'no hole'),
 ('V-indep', -0.942477796077): (washer(X**2, sp.sqrt(X), X, 0, 1), 'R and r swapped'),
 ('V-indep', 0.3): (sp.integrate(X - X**4, (X, 0, 1)), 'pi forgotten'),
 ('V-indep', 0.333333333333): (sp.integrate(sp.sqrt(X) - X**2, (X, 0, 1)), 'the area of the flat region'),
 ('V-p1', 12.566370614359): (sp.pi*sp.integrate(2*X, (X, 0, 2)), 'radius not squared'),
 ('V-p1', 10.666666666667): (sp.integrate((2*X)**2, (X, 0, 2)), 'pi forgotten'),
 ('V-p4', 0.418879020479): (washer(X, X**2, X, 0, 1), 'measured from the x-axis instead of y = 2'),
 ('V-p5', -0.523598775598): (washer(Y, sp.sqrt(Y), Y, 0, 1), 'R and r swapped'),
 ('V-p7', 1.570796326795): (sp.pi*sp.integrate(X, (X, 0, 1)), 'the diameter used as the radius'),
 ('V-p10', 28.483773392547): (washer(4*X - X**2, 3, X, 1, 3), 'spun about the x-axis instead of y = 6'),
 ('V-m4', 56.548667764616): (sp.pi*sp.integrate(9 - X**2, (X, 0, 3)), 'half the ball'),
 ('M-V1', 5.333333333333): (sp.integrate(sp.sqrt(X), (X, 0, 4)), 'the area of the flat region'),
 ('M-V3', 3.14159265359): (sp.pi*improper(sp.exp(-X), X, 0), 'radius not squared'),
 ('F-V2', 25.132741228718): (washer(sp.sqrt(X), 0, X, 0, 4), 'spun about the x-axis instead of y = 2'),
 ('F-V2', -41.887902047864): (washer(2 - sp.sqrt(X), 2, X, 0, 4), 'R and r swapped'),
}
NEW_VIDS = [v_ for v_ in KEYS if v_ in VERIFIED]
html_wrong = {(v_, float(w_['v'])) for v_ in NEW_VIDS for w_ in KEYS[v_]['wrong'] if 'v' in w_}
bad_w = []
for (vid, val), (expr, desc) in MIST.items():
    mv = float(sp.N(expr, 20))
    ok = abs(mv - val) < 1e-9 * max(1, abs(val)) and abs(val - VERIFIED[vid]) > 1e-6
    bad_w += [] if ok else [(vid, val, mv)]
    print(f"{'OK      ' if ok else 'MISMATCH'} {vid:<13} shown {val:<17} = {desc:<44} recomputed {mv:.12g}")
print(f"\n{len(MIST)} wrong values checked | every wrong value in the new answers is covered: {html_wrong == set(MIST)} | uncovered: {sorted(html_wrong - set(MIST)) or 'none'} | mismatches: {bad_w or 'none'}")

OK       E-p11         shown 27.930697094774   = |displacement|                               recomputed 27.9306970948
OK       E-p11         shown 44.181121736571   = speed taken as the sum of the components     recomputed 44.1811217366
OK       L-guided-val  shown 18.98858179594    = x-limits 4, 11 put into (2/3)u^(3/2)         recomputed 18.9885817959
OK       L-indep       shown 65.333333333333   = forgot du = 4 dx                             recomputed 65.3333333333
OK       L-indep       shown 11.897275564511   = g'^2 taken as 2(x + 2)                       recomputed 11.8972755645
OK       L-p1          shown 6.28318530718     = the whole circle                             recomputed 6.28318530718
OK       L-p1          shown 2.0               = the chord from (1, 0) to (-1, 0)             recomputed 2
OK       L-p2          shown 5.363034122669    = f(pi) taken as 0                             recomputed 5.36303412267
OK       L-p2          shown 15.168951183496   = prefactor d

## 5. Numbers and identities quoted in explanations, tryouts, spot-the-error items, figures and "What to notice" lists

In [7]:
sin_true, cos_true = true_sin, mq(lambda s: mp.sqrt(1 + mp.sin(s)**2), [0, mp.pi])
CL = [   # (where it appears, the number as shown, independent value, tolerance implied by the rounding shown)
 ('L guided step 6: chord for g on [4, 11]', 24.659, sp.sqrt(49 + (g.subs(X, 11) - g.subs(X, 4))**2), 1e-3),
 ('L what-to-notice: n = 1 within 0.01 of 74/3', 0.0074, sp.Rational(74, 3) - sp.sqrt(49 + (g.subs(X, 11) - g.subs(X, 4))**2), 1e-4),
 ('L guided step 6: slope from sqrt 8', 2.83, sp.sqrt(8), 5e-3), ('L guided step 6: slope to sqrt 15', 3.87, sp.sqrt(15), 5e-3),
 ('L tryout 1: int sqrt(1 + cos x) = 4 sqrt 2', 5.66, sp.integrate(sp.sqrt(2)*sp.Abs(sp.cos(X/2)), (X, 0, 2*sp.pi)), 5e-3),
 ('L tryout 1: int (1 + cos x) = 2 pi', 6.28, sp.integrate(1 + sp.cos(X), (X, 0, 2*sp.pi)), 5e-3),
 ('L tryout 1 / compare: actual length of sin', 7.64, sin_true, 5e-3),
 ('L practice 2: actual length of sin', 7.6404, sin_true, 5e-5),
 ('L spot 1: (pi/4)(sqrt2 + 2 + 0 + 2 + sqrt2)', 5.36, sp.pi/4*(2*sp.sqrt(2) + 4), 5e-3),
 ('L spot 2: int (1 + sqrt(x + 4)) on [4, 11]', 30.64, sp.integrate(1 + sp.sqrt(X + 4), (X, 4, 11)), 5e-3),
 ('L independent: chord', 16.32, sp.sqrt(16 + (sp.Rational(4, 3)*6**sp.Rational(3, 2) - sp.Rational(4, 3)*2**sp.Rational(3, 2))**2), 5e-3),
 ('L practice 7: height at 1/4', 1.017427, sp.sqrt(1 + sp.Rational(9, 256)), 1e-6),
 ('L practice 7: height at 3/4', 1.961545, sp.sqrt(1 + sp.Rational(729, 256)), 1e-6),
 ('L practice 7: high-accuracy value', 1.547866, mq(lambda s: mp.sqrt(1 + 9*s**4), [0, 1]), 1e-6),
 ('L practice 7/8: chord sqrt 2', 1.414, sp.sqrt(2), 5e-4),
 ('L tryout 2: (cos 2t, sin 2t), 0..pi', 2*sp.pi, arc_param([sp.cos(2*T), sp.sin(2*T)], 0, sp.pi), 1e-12),
 ('L tryout 2: (cos 2t, sin 2t), 0..pi/2', sp.pi, arc_param([sp.cos(2*T), sp.sin(2*T)], 0, sp.pi/2), 1e-12),
 ('L explorer note: circle twice', 12.5664, 4*sp.pi, 5e-5),
 ('mixed L2: actual length of cos on [0, pi]', 3.8202, cos_true, 5e-5),
 ('mixed L2: half of the sin length', float(sin_true)/2, cos_true, 1e-20),
 ('F-L3: chord from (1,0) to (0,1)', 1.414, sp.sqrt(2), 5e-4),
 ('E what-to-notice: |displacement|, notes p. 1', 27.93, MIST[('E-p11', 27.930697094774)][0], 5e-3),
 ('V what-to-notice/tryout: cone 32 pi/3', 33.51, 32*sp.pi/3, 5e-3), ('V what-to-notice: notes p. 7 volume 8 pi', 25.13, 8*sp.pi, 5e-3),
 ('V tryout 1: pi (R - r)^2', 8.38, sp.pi*sp.integrate(X**2, (X, 0, 2)), 5e-3), ('V tryout 1: area of S', 2, sp.integrate(2*X - X, (X, 0, 2)), 1e-12),
 ('V tryout 2: axis y = 5', 37.70, washer(5 - X, 5 - 2*X, X, 0, 2), 5e-3), ('V tryout 2: swapped', -37.70, washer(5 - 2*X, 5 - X, X, 0, 2), 5e-3),
 ('V tryout 2: x-axis radii', 8*sp.pi, washer(2*X, X, X, 0, 2), 1e-12),
 ('V explorer: axis y = -1 for the notes p. 7 region', 12*sp.pi, washer(2*X + 1, X + 1, X, 0, 2), 1e-12),
 ('V understand: ring R = 2, r = 1', 3*sp.pi, sp.pi*(2**2 - 1**2), 1e-12), ('V understand: pi (R - r)^2 for it', sp.pi, sp.pi*(2 - 1)**2, 1e-12),
 ('V quick check: R = 3, r = 2', 5*sp.pi, sp.pi*(3**2 - 2**2), 1e-12),
 ('V independent sanity: no-hole solid', 1.571, sp.pi*sp.integrate(X, (X, 0, 1)), 5e-4), ('V independent sanity: the hole', sp.pi/5, sp.pi*sp.integrate(X**4, (X, 0, 1)), 1e-12),
 ('V m4: 36 pi = (4/3) pi 3^3', 36*sp.pi, sp.Rational(4, 3)*sp.pi*27, 1e-12),
 ('figure coneslices caption', 33.51, sp.pi*sp.integrate((2*X)**2, (X, 0, 2)), 5e-3),
]
bad_cl = []
for where, shown, val, tol in CL:
    s_, v_ = float(sp.N(shown, 20)), float(sp.N(val, 20)) if not isinstance(val, mp.mpf) else float(val)
    ok = abs(s_ - v_) <= tol * max(1, abs(v_)) if tol >= 1e-6 else abs(s_ - v_) <= tol + 1e-15
    bad_cl += [] if ok else [where]
    print(f"{'OK      ' if ok else 'MISMATCH'} {where:<50} shown {s_:<12.7g} value {v_:.10g}")
IDS = [   # symbolic identities used in solutions and hints
 ('L practice 4: 1 + f^2 = (x^2/4 + 1/x^2)^2', 1 + sp.diff(f7, X)**2 - (X**2/4 + 1/X**2)**2),
 ('L mastery 4: 1 + y^2 = (x + 1/(4x))^2', 1 + sp.diff(X**2/2 - sp.log(X)/4, X)**2 - (X + 1/(4*X))**2),
 ('mixed L1: 1 + f^2 = (x^2 + 1)^2', 1 + sp.diff(fm, X)**2 - (X**2 + 1)**2),
 ('E practice 11: t^2 + 8t + 16 = (t + 4)^2', sp2 - (T + 4)**2),
 ('V practice 4: expansion', sp.expand((2 - X**2)**2 - (2 - X)**2) - (4*X - 5*X**2 + X**4)),
 ('V practice 10: x^2 - 4x + 6 = w^2 + 2 with w = x - 2', sp.expand((X**2 - 4*X + 6).subs(X, W + 2)) - (W**2 + 2)),
 ('V practice 10: 9 - (w^2 + 2)^2 = 5 - 4w^2 - w^4', sp.expand(9 - (W**2 + 2)**2) - (5 - 4*W**2 - W**4)),
 ('mixed V2: expansion', sp.expand((3*X - X**2 + 2)**2 - (X + 2)**2) - (X**4 - 6*X**3 + 4*X**2 + 8*X)),
 ('F-V2: 4 - (2 - sqrt x)^2 = 4 sqrt x - x', sp.expand(4 - (2 - sp.sqrt(X))**2) - (4*sp.sqrt(X) - X)),
 ('V practice 10: pi * int_{-1}^{1}(5 - 4w^2 - w^4) = 104 pi/15', sp.pi*sp.integrate(5 - 4*W**2 - W**4, (W, -1, 1)) - 104*sp.pi/15),
]
bad_id = [n_ for n_, e_ in IDS if sp.simplify(e_) != 0]
for n_, e_ in IDS: print(f"{'OK      ' if sp.simplify(e_) == 0 else 'MISMATCH'} {n_}")
div_ok = improper(X**sp.Rational(-3, 4), X, 1) == sp.oo and improper(1/X, X, 1) == sp.oo
print("claims that the flat regions of F-V3 and Gabriel's horn have infinite area (both integrals diverge):", div_ok)
print(f"\n{len(CL)} quoted numbers, {len(IDS)} identities | mismatches: {bad_cl + bad_id or 'none'}")

OK       L guided step 6: chord for g on [4, 11]            shown 24.659       value 24.65929371
OK       L what-to-notice: n = 1 within 0.01 of 74/3        shown 0.0074       value 0.007372957981
OK       L guided step 6: slope from sqrt 8                 shown 2.83         value 2.828427125
OK       L guided step 6: slope to sqrt 15                  shown 3.87         value 3.872983346
OK       L tryout 1: int sqrt(1 + cos x) = 4 sqrt 2         shown 5.66         value 5.656854249
OK       L tryout 1: int (1 + cos x) = 2 pi                 shown 6.28         value 6.283185307
OK       L tryout 1 / compare: actual length of sin         shown 7.64         value 7.640395578
OK       L practice 2: actual length of sin                 shown 7.6404       value 7.640395578
OK       L spot 1: (pi/4)(sqrt2 + 2 + 0 + 2 + sqrt2)        shown 5.36         value 5.363034123
OK       L spot 2: int (1 + sqrt(x + 4)) on [4, 11]         shown 30.64        value 30.6448888
OK       L independent: chor

## 6. Explorer reference values

The browser test (`test_guide.py`) compares the explorers' live numbers with independent mpmath references. This section checks the explorers' built-in exact values.
- **Volume explorer.** It computes the exact volume as $V(c)=\pi\,|I_2-2c\,I_1|$, where $I_1=\int(\text{up}-\text{lo})$ and $I_2=\int(\text{up}^2-\text{lo}^2)$, for any axis $y=c$ or $x=c$ outside the region. This is the washer integral rewritten, not a new method; it's only used inside the explorer, not taught. Here it is compared with direct quadrature of $\pi(R^2-r^2)$, with $R$ and $r$ measured from the axis.
- **Arc-length explorer.** Its exact values are checked by quadrature.
- **Path explorer.** The distance function for the notes p. 1 preset is checked against the integral of the speed.

In [8]:
PRES = {   # key: (up, lo, a, b, I1, I2, region's extent (min lo, max up))
 'notes p. 6':    (lambda s: 2*s, lambda s: 0*s, 0, 2, 4, sp.Rational(32, 3), 0, 4),
 'notes p. 7':    (lambda s: 2*s, lambda s: s, 0, 2, 2, 8, 0, 4),
 'Ex. 3.5.7':     (lambda s: s**2, lambda s: 0*s, 0, 2, sp.Rational(8, 3), sp.Rational(32, 5), 0, 4),
 'Ex. 3.5.8/9':   (lambda s: s, lambda s: s**2, 0, 1, sp.Rational(1, 6), sp.Rational(2, 15), 0, 1),
 'Ex. 3.5.10':    (lambda s: mp.sqrt(s), lambda s: s, 0, 1, sp.Rational(1, 6), sp.Rational(1, 6), 0, 1),
}
SYM = {'notes p. 6': (2*X, 0*X, X), 'notes p. 7': (2*X, X, X), 'Ex. 3.5.7': (X**2, 0*X, X), 'Ex. 3.5.8/9': (X, X**2, X), 'Ex. 3.5.10': (sp.sqrt(Y), Y, Y)}
worst, nP = 0.0, 0
for key, (up, lo, a, b, I1, I2, m, M) in PRES.items():
    su, sl, v = SYM[key]
    assert sp.simplify(sp.integrate(su - sl, (v, a, b)) - I1) == 0 and sp.simplify(sp.integrate(su**2 - sl**2, (v, a, b)) - I2) == 0, key
    for c in [m - 3, m - 2, m - 1.5, m - 1, m - 0.5, m, M, M + 0.5, M + 1, M + 2, M + 3]:
        cc = mp.mpf(c)
        direct = mq(lambda s: mp.pi*(max(abs(up(s) - cc), abs(lo(s) - cc))**2 - min(abs(up(s) - cc), abs(lo(s) - cc))**2), [a, b])
        worst = max(worst, abs(float(direct) - float(sp.pi*abs(I2 - 2*sp.nsimplify(c)*I1)))); nP += 1
print(f"volume explorer: exact formula vs direct quadrature on {nP} preset/axis combinations; worst difference {worst:.2e}")
ARC = [('g on [4, 11]', 74/3, mq(lambda s: mp.sqrt(s + 5), [4, 11])), ('semicircle', float(mp.pi), mq(lambda s: 1, [0, mp.pi])),
       ('sin on [0, 2 pi]', 7.640395578055424, true_sin), ('Ex. 3.3.15', 52/3, mq(lambda s: mp.sqrt(1 + s), [0, 8])),
       ('Suggested Ex. 7', 13/12, mq(lambda s: mp.sqrt(1 + (s**2/4 - 1/s**2)**2), [1, 2])), ('circle twice', float(4*mp.pi), mq(lambda s: 1, [0, 4*mp.pi]))]
arc_ok = True
for name, shown, ref in ARC:
    ok = abs(shown - float(ref)) < 1e-12; arc_ok &= ok
    print(f"   arc explorer, {name:<16}: built-in {shown:.15g}  quadrature {mp.nstr(ref, 15)}  {'OK' if ok else 'MISMATCH'}")
dist_ok = all(abs((s**2/2 + 4*s - 4.5) - float(mq(lambda z: z + 4, [1, s]))) < 1e-12 for s in [1.5, 2.7, 4.0, 5.0])
print("path explorer, notes p. 1 preset: distance t^2/2 + 4t - 9/2 equals the integral of the speed t + 4:", dist_ok)

volume explorer: exact formula vs direct quadrature on 55 preset/axis combinations; worst difference 0.00e+00
   arc explorer, g on [4, 11]    : built-in 24.6666666666667  quadrature 24.6666666666667  OK
   arc explorer, semicircle      : built-in 3.14159265358979  quadrature 3.14159265358979  OK
   arc explorer, sin on [0, 2 pi]: built-in 7.64039557805542  quadrature 7.64039557805542  OK
   arc explorer, Ex. 3.3.15      : built-in 17.3333333333333  quadrature 17.3333333333333  OK
   arc explorer, Suggested Ex. 7 : built-in 1.08333333333333  quadrature 1.08333333333333  OK
   arc explorer, circle twice    : built-in 12.5663706143592  quadrature 12.5663706143592  OK
path explorer, notes p. 1 preset: distance t^2/2 + 4t - 9/2 equals the integral of the speed t + 4: True


## 7. Wolfram Language cross-check

The outputs below were computed by Wolfram Language from the problem statements, not from the answers, and are pasted verbatim. There are three batches:
1. every new answer, keyed by the guide's own answer IDs, plus the lecture's worked answers;
2. the wrong-answer values and quoted numbers;
3. the typed expressions, the volume explorer's exact formula, and the arc-length explorer's values.

In [9]:
WOLF_ANSWERS = r"""E-p11 | 28 | 28.00000000000000
L-guided-val | 74/3 | 24.66666666666667
L-indep | 49/3 | 16.33333333333333
L-p1 | Pi | 3.141592653589793
L-p2 | (1 + Sqrt[2])*Pi | 7.584475591748159
L-p3 | 52/3 | 17.33333333333333
L-p4 | 13/12 | 1.083333333333333
L-p5-a | 6*Pi | 18.84955592153876
L-p5-b | 12*Pi | 37.69911184307752
L-p6 | 4*Sqrt[10] | 12.64911064067352
L-p7 | (14 + 4*Sqrt[10] + Sqrt[265] + Sqrt[985])/48 | 1.548180018535909
L-p8 | (-8 + 13*Sqrt[13])/27 | 1.439709873371550
L-m3-a | 3*Pi | 9.424777960769380
L-m3-b | 2*Pi | 6.283185307179586
L-m4 | (-1 + 2*E^2)/4 | 3.444528049465325
F-L2 | 14/3 | 4.666666666666667
F-L3 | 3/2 | 1.500000000000000
M-L1 | 12 | 12.00000000000000
M-L2 | ((1 + Sqrt[2] + Sqrt[6])*Pi)/4 | 3.819943643179836
V-guided-val | 8*Pi | 25.13274122871835
V-indep | (3*Pi)/10 | 0.9424777960769380
V-p1 | (32*Pi)/3 | 33.51032163829113
V-p2 | (32*Pi)/5 | 20.10619298297468
V-p3 | (2*Pi)/15 | 0.4188790204786391
V-p4 | (8*Pi)/15 | 1.675516081914556
V-p5 | Pi/6 | 0.5235987755982989
V-p6 | 1/5 | 0.2000000000000000
V-p7 | Pi/8 | 0.3926990816987242
V-p8-a | Pi/3 | 1.047197551196598
V-p8-b | Pi/9 | 0.3490658503988659
V-p9 | 58 | 58.00000000000000
V-p10 | (104*Pi)/15 | 21.78170906488923
V-m3 | Pi | 3.141592653589793
V-m4 | 36*Pi | 113.0973355292326
F-V2 | (40*Pi)/3 | 41.88790204786391
F-V3 | 2*Pi | 6.283185307179586
M-V1 | 8*Pi | 25.13274122871835
M-V2 | (136*Pi)/15 | 28.48377339254746
M-V3 | Pi/2 | 1.570796326794897
n3-p1-dist | 28 | 28.00000000000000
n3-p3-g | 74/3 | 24.66666666666667
n3-p3-semi | Pi | 3.141592653589793
n3-p4-T4 | (1 + Sqrt[2])*Pi | 7.584475591748159
n3-p6-cone | (32*Pi)/3 | 33.51032163829113
n3-p6-coneformula | (32*Pi)/3 | 33.51032163829113
n3-p7-washer | 8*Pi | 25.13274122871835
n3-p7-cones | 8*Pi | 25.13274122871835"""
WOLF_WRONG = r"""W-E-p11-displen | (4*Sqrt[450 - 5*Sqrt[5]])/3 | 27.93069709477373
W-E-p11-sum | 24 + 4*Sqrt[7] + (2*Sqrt[2]*(-1 + 5*Sqrt[5]))/3 | 44.18112173657090
W-L-guided-xlimits | (2*(-8 + 11*Sqrt[11]))/3 | 18.98858179593960
W-L-indep-factor4 | 196/3 | 65.33333333333333
W-L-indep-nosquare | (-5*Sqrt[5] + 13*Sqrt[13])/3 | 11.89727556451097
W-L-p2-fpi0 | (1 + 1/Sqrt[2])*Pi | 5.363034122668976
W-L-p2-dx | 2*(1 + Sqrt[2])*Pi | 15.16895118349632
W-L-p2-abscos | Pi | 3.141592653589793
W-L-p7-trap | (28 + 8*Sqrt[10] + Sqrt[265] + Sqrt[985])/64 | 1.577527367662456
W-L-p8-split | 2 | 2.000000000000000
W-V-guided-Rminusr | (8*Pi)/3 | 8.377580409572782
W-V-guided-nohole | (32*Pi)/3 | 33.51032163829113
W-V-guided-nopi | 8 | 8.000000000000000
W-V-indep-Rminusr | (9*Pi)/70 | 0.4039190554615448
W-V-indep-nohole | Pi/2 | 1.570796326794897
W-V-indep-swapped | (-3*Pi)/10 | -0.9424777960769380
W-V-indep-area | 1/3 | 0.3333333333333333
W-V-p1-noSquare | 4*Pi | 12.56637061435917
W-V-p1-nopi | 32/3 | 10.66666666666667
W-V-p7-diam | Pi/2 | 1.570796326794897
W-V-p10-xaxis | (136*Pi)/15 | 28.48377339254746
W-V-m4-half | 18*Pi | 56.54866776461628
W-M-V1-area | 16/3 | 5.333333333333333
W-M-V3-nosquare | Pi | 3.141592653589793
W-F-V2-xaxis | 8*Pi | 25.13274122871835
W-F-V2-swapped | (-40*Pi)/3 | -41.88790204786391
W-claim-L-chord-g | Sqrt[15989 - 1920*Sqrt[30]]/3 | 24.65929370868519
W-claim-L-chord-indep | (4*Sqrt[233 - 48*Sqrt[3]])/3 | 16.32239422724736
W-claim-L-tryout-sqrt1pcos | 4*Sqrt[2] | 5.656854249492380
W-claim-L-tryout-1pcos | 2*Pi | 6.283185307179586
W-claim-L-spot2 | 7 - (32*Sqrt[2])/3 + 10*Sqrt[15] | 30.64488879676115
W-claim-L-tryout2-b | 2*Pi | 6.283185307179586
W-claim-L-tryout2-c | Pi | 3.141592653589793
W-claim-L-m4 | (-1 + 2*E^2)/4 | 3.444528049465325
W-claim-M-L2-true | 3.82019778902771201790476208031897286667`25. | 3.820197789027712
W-claim-V-tryout-y5-swapped | -12*Pi | -37.69911184307752
W-claim-V-tryout-area | 2 | 2.000000000000000
W-claim-V-axis-m1 | 12*Pi | 37.69911184307752
W-claim-V-g7 | 8*Pi | 25.13274122871835
W-claim-V-p4-expand | x*(4 - 5*x + x^3) | symbolic
W-claim-M-V2-expand | x*(8 + 4*x - 6*x^2 + x^3) | symbolic
W-claim-F-V2-A | 4*Sqrt[x] - x | symbolic
W-claim-F-V3-area | Integrate[x^(-3/4), {x, 1, Infinity}] | symbolic
W-claim-V-m3-area | Integrate[x^(-1), {x, 1, Infinity}] | symbolic
W-claim-V-p10-w | 2 + w^2 | symbolic
W-claim-L-p4-square | 0 | 0
W-claim-M-L1-square | 0 | 0"""
WOLF_WIDGETS = r"""X-L-guided-dg | 0
X-L-guided-inside | 0
X-V-guided-R | 0
X-V-guided-r | 0
X-V-guided-A | 0
"p6 c=-2" | 4 | 32/3 | (80*Pi)/3 | True
"p6 c=-1" | 4 | 32/3 | (56*Pi)/3 | True
"p6 c=0" | 4 | 32/3 | (32*Pi)/3 | True
"p6 c=4" | 4 | 32/3 | (64*Pi)/3 | True
"p6 c=5" | 4 | 32/3 | (88*Pi)/3 | True
"p7 c=-2" | 2 | 8 | 16*Pi | True
"p7 c=-1" | 2 | 8 | 12*Pi | True
"p7 c=0" | 2 | 8 | 8*Pi | True
"p7 c=4" | 2 | 8 | 8*Pi | True
"p7 c=5" | 2 | 8 | 12*Pi | True
"x2 c=-2" | 8/3 | 32/5 | (256*Pi)/15 | True
"x2 c=-1" | 8/3 | 32/5 | (176*Pi)/15 | True
"x2 c=0" | 8/3 | 32/5 | (32*Pi)/5 | True
"x2 c=4" | 8/3 | 32/5 | (224*Pi)/15 | True
"x2 c=5" | 8/3 | 32/5 | (304*Pi)/15 | True
"358 c=-2" | 1/6 | 2/15 | (4*Pi)/5 | True
"358 c=-1" | 1/6 | 2/15 | (7*Pi)/15 | True
"358 c=0" | 1/6 | 2/15 | (2*Pi)/15 | True
"358 c=1" | 1/6 | 2/15 | Pi/5 | True
"358 c=2" | 1/6 | 2/15 | (8*Pi)/15 | True
"3510 c=-2" | 1/6 | 1/6 | (5*Pi)/6 | True
"3510 c=-1" | 1/6 | 1/6 | Pi/2 | True
"3510 c=0" | 1/6 | 1/6 | Pi/6 | True
"3510 c=1" | 1/6 | 1/6 | Pi/6 | True
"3510 c=2" | 1/6 | 1/6 | Pi/2 | True
arc-g | 74/3 | 24.66666666666667
arc-semi | Pi | 3.141592653589793
arc-sin | 7.64039557805542403580952416063794573334`25. | 7.640395578055424
arc-3315 | 52/3 | 17.33333333333333
arc-sug7 | 13/12 | 1.083333333333333
arc-twice | 4*Pi | 12.56637061435917"""
parse = lambda raw: {ln.split(' | ')[0].strip('"'): [p_.strip() for p_ in ln.split(' | ')[1:]] for ln in raw.strip().splitlines()}
WA, WW, WX = parse(WOLF_ANSWERS), parse(WOLF_WRONG), parse(WOLF_WIDGETS)
WOLFRAM, dis = {}, []
for vid, parts_ in WA.items():
    WOLFRAM[vid] = float(parts_[-1])
    if vid in VERIFIED and abs(WOLFRAM[vid] - VERIFIED[vid]) > 1e-12 * max(1, abs(VERIFIED[vid])): dis.append(vid)
print(f"answers: Wolfram values for {sum(v_ in VERIFIED for v_ in WA)} of the {len(VERIFIED)} verified values | disagreements: {dis or 'none'} | without Wolfram: {sorted(set(VERIFIED) - set(WOLFRAM)) or 'none'}")
WMAP = {'W-E-p11-displen': ('E-p11', 27.930697094774), 'W-E-p11-sum': ('E-p11', 44.181121736571), 'W-L-guided-xlimits': ('L-guided-val', 18.98858179594),
        'W-L-indep-factor4': ('L-indep', 65.333333333333), 'W-L-indep-nosquare': ('L-indep', 11.897275564511), 'W-L-p2-fpi0': ('L-p2', 5.363034122669),
        'W-L-p2-dx': ('L-p2', 15.168951183496), 'W-L-p2-abscos': ('L-p2', 3.14159265359), 'W-L-p7-trap': ('L-p7', 1.577527367662), 'W-L-p8-split': ('L-p8', 2.0),
        'W-V-guided-Rminusr': ('V-guided-val', 8.377580409573), 'W-V-guided-nohole': ('V-guided-val', 33.510321638291), 'W-V-guided-nopi': ('V-guided-val', 8.0),
        'W-V-indep-Rminusr': ('V-indep', 0.403919055462), 'W-V-indep-nohole': ('V-indep', 1.570796326795), 'W-V-indep-swapped': ('V-indep', -0.942477796077),
        'W-V-indep-area': ('V-indep', 0.333333333333), 'W-V-p1-noSquare': ('V-p1', 12.566370614359), 'W-V-p1-nopi': ('V-p1', 10.666666666667),
        'W-V-p7-diam': ('V-p7', 1.570796326795), 'W-V-p10-xaxis': ('V-p10', 28.483773392547), 'W-V-m4-half': ('V-m4', 56.548667764616),
        'W-M-V1-area': ('M-V1', 5.333333333333), 'W-M-V3-nosquare': ('M-V3', 3.14159265359), 'W-F-V2-xaxis': ('F-V2', 25.132741228718), 'W-F-V2-swapped': ('F-V2', -41.887902047864)}
# wrong values equal to a Wolfram-verified answer elsewhere (same number, different problem) or trivial constants
EQUIV = {('L-p1', 6.28318530718): WOLFRAM['L-m3-b'], ('L-p5-b', 18.849555921539): WOLFRAM['L-p5-a'], ('L-m3-b', 9.424777960769): WOLFRAM['L-m3-a'],
         ('L-m3-b', 3.14159265359): WOLFRAM['L-p1'], ('V-p4', 0.418879020479): WOLFRAM['V-p3'], ('V-p5', -0.523598775598): -WOLFRAM['V-p5'],
         ('V-indep', 0.3): WOLFRAM['V-indep'] / float(mp.pi), ('L-p1', 2.0): 2.0}
wdis = [k_ for k_, key in WMAP.items() if abs(float(WW[k_][-1]) - key[1]) > 1e-9 * max(1, abs(key[1]))]
edis = [k_ for k_, v_ in EQUIV.items() if abs(v_ - k_[1]) > 1e-9 * max(1, abs(k_[1]))]
print(f"wrong values: {len(WMAP)} computed directly by Wolfram + {len(EQUIV)} equal to Wolfram-verified numbers (or the constant 2) = {len(WMAP) + len(EQUIV)} of {len(MIST)} | disagreements: {wdis + edis or 'none'} | covered: {set(WMAP.values()) | set(EQUIV) == set(MIST)}")
claim_pairs = [('W-claim-L-chord-g', 24.65929370868519), ('W-claim-L-chord-indep', 16.32239422724736), ('W-claim-L-tryout-sqrt1pcos', float(4*mp.sqrt(2))),
               ('W-claim-L-tryout-1pcos', float(2*mp.pi)), ('W-claim-L-spot2', float(sp.N(sp.integrate(1 + sp.sqrt(X + 4), (X, 4, 11)), 20))),
               ('W-claim-M-L2-true', float(cos_true)), ('W-claim-V-tryout-y5-swapped', float(-12*mp.pi)), ('W-claim-V-axis-m1', float(12*mp.pi)), ('W-claim-V-g7', float(8*mp.pi))]
cdis = [k_ for k_, v_ in claim_pairs if abs(float(WW[k_][-1]) - v_) > 1e-12 * max(1, abs(v_))]
sym_ok = WW['W-claim-L-p4-square'][0] == '0' and WW['W-claim-M-L1-square'][0] == '0' and WW['W-claim-V-p10-w'][0] == '2 + w^2' and WW['W-claim-F-V2-A'][0] == '4*Sqrt[x] - x'
print(f"quoted numbers: {len(claim_pairs)} compared with Wolfram | disagreements: {cdis or 'none'} | symbolic identities agree: {sym_ok} | Wolfram also reported that both 'infinite area' integrals do not converge (Integrate::idiv)")
x_ok = all(WX[k_][0] == '0' for k_ in ['X-L-guided-dg', 'X-L-guided-inside', 'X-V-guided-R', 'X-V-guided-r', 'X-V-guided-A'])
pap_ok = all(v_[-1] == 'True' for k_, v_ in WX.items() if ' c=' in k_)
arcw_ok = all(abs(float(WX[k_][-1]) - float(ref)) < 1e-12 for k_, ref in zip(['arc-g', 'arc-semi', 'arc-sin', 'arc-3315', 'arc-sug7', 'arc-twice'], [74/3, mp.pi, true_sin, 52/3, 13/12, 4*mp.pi]))
print(f"Wolfram: typed-expression answers equal the intended expressions: {x_ok} | explorer exact-volume formula = direct integral on {sum(' c=' in k_ for k_ in WX)} combinations: {pap_ok} | arc explorer values: {arcw_ok}")

answers: Wolfram values for 47 of the 47 verified values | disagreements: none | without Wolfram: none
wrong values: 26 computed directly by Wolfram + 8 equal to Wolfram-verified numbers (or the constant 2) = 34 of 34 | disagreements: none | covered: True
quoted numbers: 9 compared with Wolfram | disagreements: none | symbolic identities agree: True | Wolfram also reported that both 'infinite area' integrals do not converge (Integrate::idiv)
Wolfram: typed-expression answers equal the intended expressions: True | explorer exact-volume formula = direct integral on 25 combinations: True | arc explorer values: True


## 8. Final pass on the built guide

- Every new embedded answer must be verified above (SymPy/mpmath and Wolfram).
- Every answer that existed before the update must be unchanged, and must still equal the value verified on 2026-09-23.
- No interactive item may be lost.

In [10]:
src = GUIDE.read_text(encoding='utf-8')
inputs = re.findall(r"""<input class="ans"((?:[^>"']|"[^"]*"|'[^']*')*)>""", src)
OLD = json.load(open(VER / 'verified_values.json', encoding='utf-8'))
BEFORE = json.load(open(VER / 'guide_state_before_2026-09-25.json', encoding='utf-8'))
before_keys = {k_['vid']: k_ for k_ in BEFORE['answer_keys']}
bad_ans, n_new, n_old = [], 0, 0
for attrs in inputs:
    vid = re.search(r'data-vid="([^"]+)"', attrs).group(1)
    ans = re.search(r'data-answer="([^"]+)"', attrs).group(1)
    kind = (re.search(r'data-kind="([^"]+)"', attrs) or [None, 'num'])[1]
    if vid in before_keys:
        n_old += 1
        if before_keys[vid]['answer'] != ans: bad_ans.append((vid, 'CHANGED', before_keys[vid]['answer'], ans))
        continue
    n_new += 1
    if kind in ('anti', 'expr'):
        if not EXPR_OK.get(vid): bad_ans.append((vid, 'expression not verified'))
        continue
    ref = VERIFIED.get(vid)
    if ref is None: bad_ans.append((vid, 'NOT VERIFIED')); continue
    if abs(float(ans) - ref) > 1e-9 * max(1, abs(ref)): bad_ans.append((vid, ans, ref))
    if vid not in WOLFRAM: bad_ans.append((vid, 'no Wolfram check'))
now_vids = {re.search(r'data-vid="([^"]+)"', a_).group(1) for a_ in inputs}
lost = sorted(set(before_keys) - now_vids)
print(f"answers embedded in the guide: {len(inputs)} = {n_old} from before (unchanged) + {n_new} new | problems: {bad_ans or 'none'} | lost: {lost or 'none'}")
old_num = [v_ for v_, k_ in before_keys.items() if k_['kind'] == 'num' and v_ in OLD and k_['answer'] != 'diverges']
old_ok = all(abs(float(before_keys[v_]['answer']) - OLD[v_]) < 1e-9 * max(1, abs(OLD[v_])) for v_ in old_num)
print(f"old numeric answers that still equal the 2026-09-23 verified values: {old_ok} ({len(old_num)} compared)")
ids_now = set(re.findall(r'\sdata-(?:pid|qid|gid|sid|eid)="([^"]+)"', src))
print("interactive items from before that are missing now:", sorted(set(BEFORE['ids']) - ids_now) or 'none', f"| items now: {len(ids_now)} (before {len(BEFORE['ids'])})")
PAT = {'answer_inputs': r'<input class="ans"', 'problems': r'class="problem"', 'mcq': r'class="mcq"', 'spots': r'class="spot"', 'tryouts': r'class="tryout"',
       'guided': r'class="guided"', 'explain': r'class="explain card"', 'widgets': r'<div id="widget-', 'topic_sections': r'<section id="[A-Z]" class="topic"'}
print("component counts, before -> after:", {k_: (BEFORE['counts'][k_], len(re.findall(p_, src))) for k_, p_ in PAT.items()})

answers embedded in the guide: 219 = 175 from before (unchanged) + 44 new | problems: none | lost: none
old numeric answers that still equal the 2026-09-23 verified values: True (120 compared)
interactive items from before that are missing now: none | items now: 298 (before 237)
component counts, before -> after: {'answer_inputs': (175, 219), 'problems': (128, 162), 'mcq': (78, 97), 'spots': (14, 18), 'tryouts': (9, 13), 'guided': (8, 10), 'explain': (9, 11), 'widgets': (8, 10), 'topic_sections': (8, 10)}


## 9. Scope checks

- The new sections may mention excluded material (shells, polar, surface area, x = f(y)) only to say it's not in the notes, or as a wrong option.
- No new §1.1/§1.3 items: the counts of items tagged U, P, R must be unchanged.

In [11]:
vis = re.sub(r'<(script|style)[\s\S]*?</\1>', ' ', src)
vis = re.sub(r'<annotation[\s\S]*?</annotation>', ' ', vis)
def section_text(sid):
    m_ = re.search(r'<section id="%s" class="topic">([\s\S]*?)</section>' % sid, vis)
    return re.sub(r'\s+', ' ', _html.unescape(re.sub(r'<[^>]+>', ' ', m_.group(1)))) if m_ else ''
newtxt = section_text('L') + ' ' + section_text('V')
for term in ['shell', 'polar', 'surface area', 'x = f(y)', 'by arc length', 'hyperbolic', 'cosh', 'trig substitution', 'Pappus', 'centroid']:
    hits = [m_.start() for m_ in re.finditer(re.escape(term), newtxt, re.I)]
    print(f"   '{term}': {len(hits)} mention(s)", [newtxt[max(0, h_ - 70): h_ + 30] for h_ in hits][:4])
print("topic sections:", re.findall(r'<section id="([A-Z])" class="topic"', src))
print("items tagged with the §1.1/§1.3 topics, before -> after:", {T_: (BEFORE['topic_attr_counts'][T_], len(re.findall(r'data-topic="%s"' % T_, src))) for T_ in 'UPR'})
print("new sections and items per topic:", {T_: len(re.findall(r'data-topic="%s"' % T_, src)) for T_ in 'ABCDELV'})

   'shell': 2 mention(s) ["uggested Ex. 1, 2, 4, 7, 9; Additional Ex. 1, 9, 13, 14). Cylindrical shells aren't in your notes, s", ' a gap of height x x between the axis and the region disk cylindrical shell At position x x the regi']
   'polar': 0 mention(s) []
   'surface area': 0 mention(s) []
   'x = f(y)': 0 mention(s) []
   'by arc length': 0 mention(s) []
   'hyperbolic': 0 mention(s) []
   'cosh': 0 mention(s) []
   'trig substitution': 0 mention(s) []
   'Pappus': 0 mention(s) []
   'centroid': 0 mention(s) []
topic sections: ['U', 'P', 'R', 'A', 'B', 'C', 'D', 'E', 'L', 'V']


items tagged with the §1.1/§1.3 topics, before -> after: {'U': (21, 21), 'P': (23, 23), 'R': (22, 22)}
new sections and items per topic: {'A': 17, 'B': 26, 'C': 18, 'D': 20, 'E': 24, 'L': 21, 'V': 23}


## 10. Summary

In [12]:
bad_rows = [r_ for r_ in ROWS if not r_['ok']]
n_w = sum(v_ in WOLFRAM for v_ in VERIFIED)
print(f"SymPy/mpmath checks: {len(ROWS)} rows, {len(bad_rows)} mismatches")
print(f"Wolfram: {n_w}/{len(VERIFIED)} verified values cross-checked, {len(dis)} disagreements; {len(WMAP)} wrong values and {len(claim_pairs)} quoted numbers computed directly")
print(f"typed-expression answers verified: {sum(EXPR_OK.values())}/{len(EXPR_OK)} | targeted wrong values: {len(MIST)} | quoted numbers: {len(CL)} | identities: {len(IDS)}")
print(f"built guide: {len(inputs)} answers ({n_new} new, all verified: {not bad_ans}); nothing lost: {not lost}; old answers unchanged: {old_ok}")
assert not bad_rows and not dis and not bad_ans and not lost and old_ok and not bad_w and not bad_cl and not bad_id and all(EXPR_OK.values())
assert x_ok and pap_ok and arcw_ok and arc_ok and dist_ok and div_ok and sym_ok and not wdis and not edis and not cdis and worst < 1e-12
json.dump(VERIFIED, open(VER / 'verified_values_arclength_volume.json', 'w'), indent=1)
print("all checks passed; saved verified_values_arclength_volume.json")

SymPy/mpmath checks: 47 rows, 0 mismatches
Wolfram: 47/47 verified values cross-checked, 0 disagreements; 26 wrong values and 9 quoted numbers computed directly
typed-expression answers verified: 5/5 | targeted wrong values: 34 | quoted numbers: 37 | identities: 10
built guide: 219 answers (44 new, all verified: True); nothing lost: True; old answers unchanged: True
all checks passed; saved verified_values_arclength_volume.json
